# Blueprint Analysis — The Naylor Blueprint

Reads the Ground Covered leaderboard (built by `Data Pipeline.ipynb`) and:
1. **Computes BCS** — Blueprint Conversion Score per runner-season
2. **Per-season tables** — Top 25 / Bottom 25 BCS per year (2023–2026)
3. **Figures** — Scatter plot, Top/Bottom bar charts, per-season panels with team logos
4. **Report** — Rebuilds `Reports/Naylor Blueprint Report.docx`

**No network required** — reads from `Data Frame/Naylor Blueprint.xlsx` (generated by Data Pipeline).

---

In [ ]:
import sys, subprocess
from pathlib import Path

REPO = Path().resolve()
for p in [REPO] + list(REPO.parents):
    if (p / 'Computer Vision' / 'Statcast Analysis Core').exists():
        REPO = p
        break

CORE = REPO / 'Computer Vision' / 'Statcast Analysis Core'
FIGS = REPO / 'Figures'
DATA = REPO / 'Data Frame'
REPORTS = REPO / 'Reports'

sys.path.insert(0, str(CORE))
print(f'Repo: {REPO}')

## 1. Run Blueprint Model

Computes BCS for all volume-qualified runner-seasons and writes per-season Top/Bottom 25 sheets to xlsx.

In [ ]:
r = subprocess.run(
    [sys.executable, str(CORE / 'naylor_blueprint.py')],
    capture_output=True, text=True
)
print(r.stdout[-3000:] if len(r.stdout) > 3000 else r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr[:500])

In [ ]:
import pandas as pd

xlsx = DATA / 'Naylor Blueprint.xlsx'
lb = pd.read_excel(xlsx, sheet_name='Blueprint Leaderboard')
print(f'Blueprint Leaderboard: {len(lb)} rows')
print('\nTop 10:')
print(lb[['rank_BCS','player_name','team','season','sprint_pctile','SB','CS','SB_pct','BCS']].head(10).to_string(index=False))
print('\nBottom 10:')
print(lb[['rank_BCS','player_name','team','season','sprint_pctile','SB','CS','SB_pct','BCS']].tail(10).to_string(index=False))

## 2. Per-Season BCS Top/Bottom 25

In [ ]:
for yr in [2023, 2024, 2025, 2026]:
    top = pd.read_excel(xlsx, sheet_name='BCS Top 25 by Season')
    bot = pd.read_excel(xlsx, sheet_name='BCS Bot 25 by Season')
    t = top[top['season']==yr][['year_rank','player_name','team','sprint_pctile','SB','CS','SB_pct','BCS']].head(5)
    b = bot[bot['season']==yr][['year_rank','player_name','team','sprint_pctile','SB','CS','SB_pct','BCS']].head(5)
    partial = ' (partial season — min 3 attempts)' if yr == 2026 else ''
    print(f'\n=== {yr}{partial} ===')
    print(f'TOP 5:')
    print(t.to_string(index=False))
    print(f'BOT 5:')
    print(b.to_string(index=False))

## 3. Generate Per-Season Figures with Team Logos

In [ ]:
# Per-season BCS Top/Bottom 25 figures
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image

LOGOS = FIGS / 'logos'
NAYLOR_ID, SOTO_ID = 647304, 665742

def load_logo(team, size=18):
    p = LOGOS / f'{team}.png'
    if not p.exists(): return None
    try:
        img = Image.open(p).convert('RGBA').resize((size, size), Image.LANCZOS)
        return np.array(img)
    except: return None

def make_season_bars(df_all, mode='top', n=25, fname='fig.png', title=''):
    seasons = [2023, 2024, 2025, 2026]
    fig, axes = plt.subplots(1, 4, figsize=(26, 13))
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.01)

    for ax, yr in zip(axes, seasons):
        sub = df_all[df_all['season'] == yr].head(n).copy()
        sub = sub.sort_values('BCS', ascending=(mode == 'bot')).reset_index(drop=True)

        bcs_vals = sub['BCS'].values
        y = np.arange(len(sub))
        colors = ['crimson' if r == NAYLOR_ID else
                  'darkgreen' if r == SOTO_ID else
                  '#3e8c5c' if v >= 0 else '#b03030'
                  for r, v in zip(sub['runner_id'].values, bcs_vals)]

        ax.barh(y, bcs_vals, color=colors, alpha=0.85, height=0.75)
        labels = [f"{r['player_name'].split()[-1]}  {int(r['SB'])}/{int(r['CS'])}" for _, r in sub.iterrows()]
        ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=7.5)

        xmin = bcs_vals.min() - 0.5
        for i, (_, r) in enumerate(sub.iterrows()):
            logo = load_logo(r['team'])
            if logo is not None:
                ab = AnnotationBbox(OffsetImage(logo, zoom=1.0), (xmin - 0.1, i),
                                    xycoords='data', frameon=False, box_alignment=(1.0, 0.5))
                ax.add_artist(ab)

        for i, v in enumerate(bcs_vals):
            ax.text(v + (0.05 if v >= 0 else -0.05), i, f'{v:+.2f}',
                    va='center', fontsize=6.0, ha='left' if v >= 0 else 'right')

        ax.axvline(0, color='black', lw=0.8)
        ax.set_title(f'{yr}{"†" if yr==2026 else ""}', fontsize=12, fontweight='bold')
        ax.set_xlabel('BCS', fontsize=9); ax.grid(axis='x', alpha=0.2); ax.margins(x=0.22)

    fig.text(0.5, -0.015,
             '† 2026: partial season (~1/3 complete as of May 2026); min 3 tracked attempts.  Red=Naylor | Green=Soto',
             ha='center', fontsize=8.5, style='italic')
    fig.tight_layout()
    fig.savefig(FIGS / fname, dpi=130, bbox_inches='tight')
    plt.close(fig)
    print(f'[write] {FIGS / fname}')

lb_full = pd.read_excel(xlsx, sheet_name='Blueprint Leaderboard')
top_df = pd.read_excel(xlsx, sheet_name='BCS Top 25 by Season').merge(
    lb_full[['player_name','season','runner_id']], on=['player_name','season'], how='left')
bot_df = pd.read_excel(xlsx, sheet_name='BCS Bot 25 by Season').merge(
    lb_full[['player_name','season','runner_id']], on=['player_name','season'], how='left')

make_season_bars(top_df, 'top', 25, 'Fig_BCS_Top25_ByYear.png',
                 'BCS Top 25 — The Blueprint (per season 2023–2026)')
make_season_bars(bot_df, 'bot', 25, 'Fig_BCS_Bot25_ByYear.png',
                 'BCS Bottom 25 — The Squanderers (per season 2023–2026)')

## 4. Rebuild DOCX Report

Runs the Node.js build script and applies the pBdr XML fix.

In [ ]:
import os, re, shutil, tempfile, zipfile

build_js = CORE / 'build_blueprint_report.js'
docx_out = REPORTS / 'Naylor Blueprint Report.docx'

# Find Node.js skill path
skill_candidates = list(Path.home().glob(
    'Library/Application Support/Claude/local-agent-mode-sessions/skills-plugin/*/*/skills/docx'))
SKILL_DIR = skill_candidates[0] if skill_candidates else None
if SKILL_DIR is None:
    print('Could not find docx skill directory. Skipping DOCX build.')
else:
    print(f'Skill dir: {SKILL_DIR}')
    env = os.environ.copy()
    env['NODE_PATH'] = str(SKILL_DIR / 'node_modules')
    r = subprocess.run(['node', str(build_js)], capture_output=True, text=True, env=env)
    print(r.stdout)
    if r.returncode != 0:
        print('ERROR:', r.stderr[:500])
    else:
        # Apply pBdr fix
        with tempfile.TemporaryDirectory() as tmpdir:
            # unpack
            with zipfile.ZipFile(docx_out) as z:
                z.extractall(tmpdir)
            # fix document.xml
            doc_xml = Path(tmpdir) / 'word' / 'document.xml'
            xml = doc_xml.read_text(encoding='utf-8')
            def fix_pbdr(m):
                c = m.group(1)
                def ge(tag):
                    mm = re.search(rf'(<w:{tag}\b[^/]*/?>)', c)
                    return mm.group(1) if mm else ''
                parts = [ge(t) for t in ['top','left','bottom','right']]
                parts = [p for p in parts if p]
                inner = '\n          '.join(parts)
                return f'<w:pBdr>\n          {inner}\n        </w:pBdr>'
            xml_fixed = re.sub(r'<w:pBdr>(.*?)</w:pBdr>', fix_pbdr, xml, flags=re.DOTALL)
            doc_xml.write_text(xml_fixed, encoding='utf-8')
            # repack
            with zipfile.ZipFile(docx_out, 'w', zipfile.ZIP_DEFLATED) as zout:
                for f in Path(tmpdir).rglob('*'):
                    if f.is_file():
                        zout.write(f, f.relative_to(tmpdir))
        print(f'[write] {docx_out}  ({docx_out.stat().st_size // 1024}KB)')